In [29]:
import pandas as pd
pd.set_option('display.max_columns', None) # show all columns
pd.set_option('display.width', 1000)

import os
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import numpy as np
from dython.nominal import associations

from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from xgboost import XGBClassifier



data_dir = ''

def preprocess_train(df):
    ## Rename columns
    col_names = {
        'id':'id',
        'Age':'age',
        'Sex':'sex',
        'Chest pain type':'chest_pain_type',
        'BP':'bp',
        'Cholesterol':'cholesterol',
        'FBS over 120':'fbs_over_120',
        'EKG results':'ekg_results',
        'Max HR':'max_hr',
        'Exercise angina':'exercise_angina',
        'ST depression':'st_depression',
        'Slope of ST':'slope_of_st',
        'Number of vessels fluro':'number_of_vessels_fluro',
        'Thallium':'thallium',
        'Heart Disease':'heart_disease'
    }

    df = df.rename(columns=col_names)

    ## Recode target variable column to binary
    df['heart_disease'] = df['heart_disease'].replace({'Presence':1, 'Absence':0}).astype(int)


    ## Drop id column
    drop_cols = ['id']
    df = df.drop(columns=drop_cols)
    return df

def preprocess_test(df):
    ## Rename columns
    col_names = {
        'id':'id',
        'Age':'age',
        'Sex':'sex',
        'Chest pain type':'chest_pain_type',
        'BP':'bp',
        'Cholesterol':'cholesterol',
        'FBS over 120':'fbs_over_120',
        'EKG results':'ekg_results',
        'Max HR':'max_hr',
        'Exercise angina':'exercise_angina',
        'ST depression':'st_depression',
        'Slope of ST':'slope_of_st',
        'Number of vessels fluro':'number_of_vessels_fluro',
        'Thallium':'thallium',
    }

    df = df.rename(columns=col_names)

    ## Recode target variable column to binary
    # df['heart_disease'] = df['heart_disease'].replace({'Presence':1, 'Absence':0}).astype(int)


    ## Drop id column
    # drop_cols = ['id']
    # df = df.drop(columns=drop_cols)
    return df

In [2]:
df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
df = preprocess_train(df)

df_test = pd.read_csv(os.
path.join(data_dir, 'test.csv'))
df_test = preprocess_test(df_test)

df.head()

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,number_of_vessels_fluro,thallium,heart_disease
0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,1
1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,0
2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,0
3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,0
4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,1


# Augment Features

In [51]:
def augment_features(df):
    ## Nonlinear Features
    df['age^2'] = df['age'] ** 2
    df['age^3'] = df['age'] ** 3
    df['bp^2'] = df['bp'] ** 2
    df['cholesterol^2'] = df['cholesterol'] **2
    df['max_hr^2'] = df['max_hr']**2
    df['st_depression_log'] = df['max_hr'].apply(np.log1p)

    ## Ratios
    df['st_hr_index'] = df['st_depression'] / df['max_hr'] # dont actually think this is correct calculation
    df['bp_age_ratio'] = df['bp'] / df['age']
    df['cholesterol_age_ratio'] = df['cholesterol'] / df['age']
    df['max_hr_age_ratio'] = df['max_hr'] / df['age']


    ## Composite Scores
    df['ischemia_score'] = df['st_depression'] + df['exercise_angina'] + df['number_of_vessels_fluro']
    df['exercise_capacity_score'] = df['max_hr'] / (220 - df['age'])

    return df


In [9]:
## df with features
df_wf = augment_features(df)

In [10]:
df_wf

,age,sex,chest_pain_type,bp,cholesterol,fbs_over_120,ekg_results,max_hr,exercise_angina,st_depression,slope_of_st,number_of_vessels_fluro,thallium,heart_disease,age^2,age^3,bp^2,cholesterol^2,max_hr^2,st_depression_log,st_hr_index,bp_age_ratio,cholesterol_age_ratio,max_hr_age_raito,ischemia_score,exercise_capacity_score
0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,1,3364,195112,23104,57121,24964,5.068904,0.022785,2.620690,4.120690,2.724138,6.6,0.975309
1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,0,2704,140608,15625,105625,29241,5.147494,0.000000,2.403846,6.250000,3.288462,0.0,1.017857
2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,0,3136,175616,25600,35344,22801,5.023881,0.000000,2.857143,3.357143,2.696429,0.0,0.920732
3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,0,1936,85184,17956,52441,22500,5.017280,0.006667,3.045455,5.204545,3.409091,1.0,0.852273
4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,1,3364,195112,19600,54756,15625,4.836282,0.030400,2.413793,4.034483,2.155172,7.8,0.771605
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629995,56,0,1,110,226,0,0,132,0,0.0,1,0,7,0,3136,175616,12100,51076,17424,4.890349,0.000000,1.964286,4.035714,2.357143,0.0,0.804878
629996,54,1,4,128,249,1,2,150,0,0.0,2,0,3,0,2916,157464,16384,62001,22500,5.017280,0.000000,2.370370,4.611111,2.777778,0.0,0.903614
629997,67,1,4,130,275,0,0,149,0,0.0,1,2,7,1,4489,300763,16900,75625,22201,5.010635,0.000000,1.940299,4.104478,2.223881,2.0,0.973856
629998,52,1,4,140,199,0,2,157,0,0.0,1,0,6,1,2704,140608,19600,39601,24649,5.062595,0.000000,2.692308,3.826923,3.019231,0.0,0.934524


# Test run with all augmented features

Produced AUC of: 0.8853076896206943


In [83]:
dep_var_name = 'heart_disease'

X = df.drop([dep_var_name], axis=1)
y = df[dep_var_name]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

[d.shape for d in [X_train, X_test, y_train, y_test]]

[(504000, 25), (126000, 25), (504000,), (126000,)]

In [33]:
param_optimal = {
    'learning_rate':0.1,
    'n_estimators':140,
    'max_depth':6,
    'min_child_weight':4,
    'gamma':0,
    'subsample':0.65,
    'colsample_bytree':0.75,
    'objective':'binary:logistic',
    'scale_pos_weight':1,
    'seed':42    
}

xgb = XGBClassifier(**param_optimal).fit(X_train, y_train)

xgb_class = xgb.predict(X_test)

xgb_auc = roc_auc_score(y_test, xgb_class)
print(f"AUC score: {xgb_auc}")

AUC score: 0.8853076896206943


Different XGBClassifier training params I saw; seems to calculate a CV score 

For whatever reason I have to feed in dataframe with `.values` to avoid a `ValueError: feature_names mismatch:` during the predict()

In [ ]:
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'n_estimators': 2000,
    'learning_rate': 0.05,
    'max_depth': 2,
    'max_bin': 128,
    'reg_lambda': 1,
    'scale_pos_weight': 1,
    'subsample': 0.95, 
    'colsample_bytree': 0.95, 
    'grow_policy': 'depthwise',
    'tree_method': 'auto', 
    'enable_categorical': True,
    'early_stopping_rounds': 10,
    'tree_method':'hist', 
    # 'predictor': 'gpu_predictor', ## getting user warning that this param is not used
}


model = XGBClassifier(
    **xgb_params,
    verbosity=1
).fit(X_train.values, y_train.values, eval_set=[(X_train.values, y_train.values), (X_test.values, y_test.values)])





[0]	validation_0-auc:0.87536	validation_1-auc:0.87442
[1]	validation_0-auc:0.87536	validation_1-auc:0.87442
[2]	validation_0-auc:0.88504	validation_1-auc:0.88402
[3]	validation_0-auc:0.91313	validation_1-auc:0.91251


c:\Users\abhi-\miniconda3\envs\capstone2\Lib\site-packages\xgboost\callback.py:386: UserWarning: [17:41:03] WARNING: C:\miniconda3\conda-bld\xgboost-split_1764761400759\work\src\learner.cc:790: 
Parameters: { "predictor" } are not used.

  self.starting_round = model.num_boosted_rounds()


[4]	validation_0-auc:0.91673	validation_1-auc:0.91614
[5]	validation_0-auc:0.91645	validation_1-auc:0.91589
[6]	validation_0-auc:0.91645	validation_1-auc:0.91589
[7]	validation_0-auc:0.91656	validation_1-auc:0.91595
[8]	validation_0-auc:0.91656	validation_1-auc:0.91595
[9]	validation_0-auc:0.91499	validation_1-auc:0.91439
[10]	validation_0-auc:0.91499	validation_1-auc:0.91439
[11]	validation_0-auc:0.91931	validation_1-auc:0.91871
[12]	validation_0-auc:0.91931	validation_1-auc:0.91871
[13]	validation_0-auc:0.92427	validation_1-auc:0.92365
[14]	validation_0-auc:0.92422	validation_1-auc:0.92359
[15]	validation_0-auc:0.92422	validation_1-auc:0.92359
[16]	validation_0-auc:0.92987	validation_1-auc:0.92940
[17]	validation_0-auc:0.92904	validation_1-auc:0.92855
[18]	validation_0-auc:0.92979	validation_1-auc:0.92942
[19]	validation_0-auc:0.93014	validation_1-auc:0.92976
[20]	validation_0-auc:0.92978	validation_1-auc:0.92930
[21]	validation_0-auc:0.93031	validation_1-auc:0.92987
[22]	validation_

# Feature Elimination

## Remove Highly Correlated Features

In [39]:
features_cat = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results', 'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']

df_cat = df[features_cat]
df_num = df.drop(features_cat, axis=1).drop(['heart_disease'], axis=1)


dep_var_name = 'heart_disease'
y = df[dep_var_name]

In [40]:
def remove_high_corr(data, thresh):
    corr_matrix = associations(dataset=data, compute_only=True)['corr'].abs()
    upper_tri=corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [col for col in upper_tri.columns if any(upper_tri[col]>thresh)]
    data_sub = data.drop(to_drop, axis=1)
    return data_sub, to_drop

In [41]:
df_num_nhc, high_corr_cols = remove_high_corr(df_num, 0.7)

print(f"Dropped {len(high_corr_cols)} highly correlated columns")
print(high_corr_cols)

Dropped 12 highly correlated columns
['age^2', 'age^3', 'bp^2', 'cholesterol^2', 'max_hr^2', 'st_depression_log', 'st_hr_index', 'bp_age_ratio', 'cholesterol_age_ratio', 'max_hr_age_raito', 'ischemia_score', 'exercise_capacity_score']


## Remove Near Zero Variance

In [48]:
# Performing step in isolation instead of after df No Highly Correlated
vt = VarianceThreshold(threshold=0.1)
vt.fit_transform(df_num)

cols = df_num.columns[vt.get_support()]
df_num_nnzv = df_num[cols]

cols_dropped = [col for col in df_num.columns if col not in cols]
print(f"Dropped {len(cols_dropped)} columns with near zero variance")
print(cols_dropped)



# ## Performing step after df No Highly Correlated
# vt = VarianceThreshold(threshold=0.1)
# vt.fit_transform(df_num_nhc)

# cols = df_num_nhc.columns[vt.get_support()]
# df_num_nhc_nnzv = df_num_nhc[cols]

# cols_dropped = [col for col in df_num_nhc.columns if col not in cols]
# print(f"Dropped {len(cols_dropped)} columns with near zero variance")
# print(cols_dropped)

Dropped 3 columns with near zero variance
['st_depression_log', 'st_hr_index', 'exercise_capacity_score']


# Test run with reduced features

Not reducing features had higher AUC of both

In [ ]:
# df_full = pd.concat([df_num_nhc_nnzv.reset_index(drop=True), df_cat.reset_index(drop=True), df['heart_disease']], axis=1) ## --> Produced AUC of 0.884

df_full = pd.concat([df_num_nnzv.reset_index(drop=True), df_cat.reset_index(drop=True), df['heart_disease']], axis=1)       ## --> Produced AUC of 0.8850766677434503

In [53]:
dep_var_name = 'heart_disease'

X = df_full.drop([dep_var_name], axis=1)
y = df_full[dep_var_name]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


param_optimal = {
    'learning_rate':0.1,
    'n_estimators':140,
    'max_depth':6,
    'min_child_weight':4,
    'gamma':0,
    'subsample':0.65,
    'colsample_bytree':0.75,
    'objective':'binary:logistic',
    'scale_pos_weight':1,
    'seed':42    
}

xgb = XGBClassifier(**param_optimal).fit(X_train, y_train)

xgb_class = xgb.predict(X_test)

xgb_auc = roc_auc_score(y_test, xgb_class)
print(f"AUC score: {xgb_auc}")

AUC score: 0.8850766677434503


# Normalize Numeric Features

In [54]:
def scale_features(df_num):
    from sklearn.preprocessing import StandardScaler
    df_std = StandardScaler().fit_transform(df_num)
    scaled = pd.DataFrame(df_std, columns=df_num.columns)
    return scaled

In [55]:
df_num_scaled = scale_features(df_num)

# Test run with scaled numeric features

Produced AUC of: 0.885040747954212

*lower than All augmented features not scaled*

In [57]:
df_full = pd.concat([df_num_scaled.reset_index(drop=True), df_cat.reset_index(drop=True), df['heart_disease']], axis=1)

dep_var_name = 'heart_disease'

X = df_full.drop([dep_var_name], axis=1)
y = df_full[dep_var_name]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


param_optimal = {
    'learning_rate':0.1,
    'n_estimators':140,
    'max_depth':6,
    'min_child_weight':4,
    'gamma':0,
    'subsample':0.65,
    'colsample_bytree':0.75,
    'objective':'binary:logistic',
    'scale_pos_weight':1,
    'seed':42    
}

xgb = XGBClassifier(**param_optimal).fit(X_train, y_train)

xgb_class = xgb.predict(X_test)

xgb_auc = roc_auc_score(y_test, xgb_class)
print(f"AUC score: {xgb_auc}")

AUC score: 0.885040747954212


# Catboost testing

My thoughts are that if baseline catboost shows promise, theres an avenue of binning the numeric variables into categorical and trying to tune a catboost model 

In [62]:
from catboost import CatBoostClassifier

In [67]:
df_full = pd.concat([df_cat.reset_index(drop=True), df['heart_disease']], axis=1)

dep_var_name = 'heart_disease'

X = df_full.drop([dep_var_name], axis=1)
y = df_full[dep_var_name]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


cb = CatBoostClassifier(cat_features=list(df_cat.columns)).fit(X_train, y_train)

Learning rate set to 0.146849
0:	learn: 0.5792944	total: 304ms	remaining: 5m 4s
1:	learn: 0.5062719	total: 429ms	remaining: 3m 34s
2:	learn: 0.4593145	total: 510ms	remaining: 2m 49s
3:	learn: 0.4197269	total: 615ms	remaining: 2m 33s
4:	learn: 0.3859329	total: 724ms	remaining: 2m 24s
5:	learn: 0.3644625	total: 807ms	remaining: 2m 13s
6:	learn: 0.3498338	total: 931ms	remaining: 2m 12s
7:	learn: 0.3412356	total: 1.02s	remaining: 2m 6s
8:	learn: 0.3342671	total: 1.12s	remaining: 2m 3s
9:	learn: 0.3285210	total: 1.21s	remaining: 2m
10:	learn: 0.3246900	total: 1.3s	remaining: 1m 57s
11:	learn: 0.3218390	total: 1.39s	remaining: 1m 54s
12:	learn: 0.3199039	total: 1.48s	remaining: 1m 52s
13:	learn: 0.3183350	total: 1.56s	remaining: 1m 49s
14:	learn: 0.3172904	total: 1.64s	remaining: 1m 47s
15:	learn: 0.3164744	total: 1.74s	remaining: 1m 47s
16:	learn: 0.3157277	total: 1.85s	remaining: 1m 47s
17:	learn: 0.3152010	total: 1.93s	remaining: 1m 45s
18:	learn: 0.3148355	total: 2.02s	remaining: 1m 44s


In [68]:
cb_class = cb.predict(X_test)
cb_auc = roc_auc_score(y_test, cb_class)
print(f"AUC score: {cb_auc}")

AUC score: 0.8656589337166473
